In [5]:
import numpy as np
import tensorflow as tf
from datetime import date
import random

# ==========================================
# 1. 核心物理引擎：生成多格式真实日期
# ==========================================
MONTHS = ["January", "February", "March", "April", "May", "June",
          "July", "August", "September", "October", "November", "December"]

# 敌军可能使用的5种伪装阵型
FORMATS = [
    "%B %d, %Y",       # April 22, 2019
    "%d %B %Y",        # 22 April 2019 
    "%m/%d/%Y",        # 04/22/2019
    "%d-%m-%Y",        # 22-04-2019
    "%b %d, %Y"        # Apr 22, 2019 (缩写)
]
#toordinal 把这个（1000，1，1）变成一个绝对整数，然后变回来的话就是 date.fromordint() 
def random_dates_multiformat(n_dates):
    min_date = date(1000, 1, 1).toordinal()
    max_date = date(9999, 12, 31).toordinal()
    ordinals = np.random.randint(max_date - min_date, size=n_dates) + min_date
    dates = [date.fromordinal(ordinal) for ordinal in ordinals]

    x = []
    y = []
    for dt in dates:
        # 抛骰子，随机选一种格式
        fmt = random.choice(FORMATS)
        
        # 避开不同电脑语言环境导致的 %B 翻译Bug
        #dt.strftime(fmt) 是 Python 官方的“日期转字符串”指令。
        if "%B" in fmt:
            date_str = dt.strftime(fmt).replace("%B", MONTHS[dt.month - 1])
        elif "%b" in fmt:
            date_str = dt.strftime(fmt).replace("%b", MONTHS[dt.month - 1][:3])
        else:
            date_str = dt.strftime(fmt)
            
        x.append(date_str)
        y.append(dt.isoformat()) # 目标始终是 ISO 标准：YYYY-MM-DD
        #dt.isoformat()：这是 Python 自带的“绝对标准输出指令”
    return x, y
    #此时的x，是带有字母的日期，就是应该训练的数据集，y是标准的输出，我们想哟啊的格式
# ==========================================
# 2. 映射字典
# 自动提取所有可能用到的字母、数字和符号 (注意加入了斜杠/)
INPUT_CHARS = "".join(set("".join(MONTHS) + "0123456789, -/"))
#"".join(MONTHS)：把所有月份的名字（January, February...）首尾相连，变成一个极其巨大的超长字符串。同时加上后边的字符，
#最后set，只有每个不同的字符。就是字符字典
char_to_id = {char: idx + 2 for idx, char in enumerate(INPUT_CHARS)}
#将字符标上号，但是没有0和1，就是从2开始标号
char_to_id["<pad>"] = 0  # 填坑用的空气
char_to_id["<sos>"] = 1  # Start Of Sequence
id_to_char = {v: k for k, v in char_to_id.items()} # 逆向翻译本，就字面意思，数字变字符

# ==========================================
# 3. 转为张量 Tensor
def prepare_dataset(n_dates):
    x, y = random_dates_multiformat(n_dates)
    
    # 编码器输入
    X_encoder = [[char_to_id[c] for c in date_str] for date_str in x]
    
    # 解码器输入：首位必须插入 <sos> 信号。
    #X_decoder 的组成是：<sos> + 2 0 1 9 - 0 4 - 2 2 （长度 11）
    X_decoder = [[char_to_id["<sos>"]] + [char_to_id[c] for c in date_str] for date_str in y]
    
    # 解码器目标：末尾顺延补一个 <pad> 占位，保持总长度一致
    #Y_decoder 的组成是：2 0 1 9 - 0 4 - 2 2 + <pad> （长度 11）
    Y_decoder = [[char_to_id[c] for c in date_str] + [char_to_id["<pad>"]] for date_str in y]

    # Padding 补齐操作，强行把参差不齐的句子拉成正规的矩阵方阵
    X_enc_padded = tf.keras.preprocessing.sequence.pad_sequences(X_encoder, padding="post")
    X_dec_padded = tf.keras.preprocessing.sequence.pad_sequences(X_decoder, padding="post")
    Y_dec_padded = tf.keras.preprocessing.sequence.pad_sequences(Y_decoder, padding="post")
    
    return X_enc_padded, X_dec_padded, Y_dec_padded

# ==========================================
X_enc, X_dec, Y_dec = prepare_dataset(10000)

print(f"(X_enc) 形状: {X_enc.shape}")
print(f"(X_dec) 形状: {X_dec.shape}")
print(f"(Y_dec) 形状: {Y_dec.shape}")

print("-"*50)
sample_original = "".join([id_to_char[i] for i in X_enc[0] if i != 0])
sample_target = "".join([id_to_char[i] for i in Y_dec[0] if i != 0])
print(f"输入: {sample_original}")
print(f"目标: {sample_target}")

(X_enc) 形状: (10000, 18)
(X_dec) 形状: (10000, 11)
(Y_dec) 形状: (10000, 11)
--------------------------------------------------
输入: 10/18/5303
目标: 5303-10-18


In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models

# ==========================================
#自定义位置编码
class PositionEmbedding(layers.Layer):
    def __init__(self, max_len, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.pos_emb = layers.Embedding(input_dim=max_len, output_dim=embed_dim)

    def call(self, x):
        # 这里的 x 已经是真实流过的数据了！所以 tf.shape 绝对不会报错！
        seq_len = tf.shape(x)[1] 
        positions = tf.range(start=0, limit=seq_len, delta=1)
        return self.pos_emb(positions)

In [3]:
# 参数设定
vocab_size = len(char_to_id)
embed_dim = 32               
num_heads = 4                
ff_dim = 128                 
max_len = 30                 

# 实例化我们刚造好的位置编码
pos_layer = PositionEmbedding(max_len=max_len, embed_dim=embed_dim)

# ==========================================
# ⚡ 左半脑：Transformer Encoder
encoder_inputs = layers.Input(shape=(None,), name="encoder_inputs")

x_emb = layers.Embedding(vocab_size, embed_dim)(encoder_inputs)
x_pos = pos_layer(encoder_inputs) # 直接调用外挂层！
x = x_emb + x_pos                 # 完美相加

attn_output = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)(x, x)
x = layers.LayerNormalization(epsilon=1e-6)(x + attn_output) 

ffn_output = layers.Dense(ff_dim, activation="relu")(x) 
ffn_output = layers.Dense(embed_dim)(ffn_output)        
encoder_outputs = layers.LayerNormalization(epsilon=1e-6)(x + ffn_output) 

# ==========================================
# ⚡ 右半脑：Transformer Decoder
decoder_inputs = layers.Input(shape=(None,), name="decoder_inputs")

y_emb = layers.Embedding(vocab_size, embed_dim)(decoder_inputs)
y_pos = pos_layer(decoder_inputs) # Decoder 也调用同一个外挂层！
y = y_emb + y_pos

attn1 = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)(
    y, y, use_causal_mask=True
)
y = layers.LayerNormalization(epsilon=1e-6)(y + attn1) 

attn2 = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)(
    query=y, value=encoder_outputs
)
y = layers.LayerNormalization(epsilon=1e-6)(y + attn2) 

ffn_out_dec = layers.Dense(ff_dim, activation="relu")(y)
ffn_out_dec = layers.Dense(embed_dim)(ffn_out_dec)
y = layers.LayerNormalization(epsilon=1e-6)(y + ffn_out_dec) 

decoder_outputs = layers.Dense(vocab_size, activation="softmax")(y)

# ==========================================
# 🚀 战机总装与点火
# ==========================================
transformer_model = models.Model([encoder_inputs, decoder_inputs], decoder_outputs)

transformer_model.compile(loss="sparse_categorical_crossentropy",
                          optimizer="nadam",
                          metrics=["accuracy"])

print("🌌 纯血 Transformer 引擎补丁加载完毕！重新点火...")
history = transformer_model.fit([X_enc, X_dec], Y_dec, epochs=15, batch_size=64, validation_split=0.2)

2026-03-17 18:16:28.328083: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M3
2026-03-17 18:16:28.328192: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-03-17 18:16:28.328207: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
2026-03-17 18:16:28.328293: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-03-17 18:16:28.328310: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


🌌 纯血 Transformer 引擎补丁加载完毕！重新点火...
Epoch 1/15


2026-03-17 18:16:30.083248: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


125/125 ━━━━━━━━━━━━━━━━━━━━ 16s 67ms/step - accuracy: 0.5409 - loss: 1.4452 - val_accuracy: 0.7292 - val_loss: 0.7600
Epoch 2/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 7s 59ms/step - accuracy: 0.8794 - loss: 0.3721 - val_accuracy: 0.9709 - val_loss: 0.1160
Epoch 3/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 8s 60ms/step - accuracy: 0.9892 - loss: 0.0518 - val_accuracy: 0.9988 - val_loss: 0.0148
Epoch 4/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 8s 60ms/step - accuracy: 0.9995 - loss: 0.0086 - val_accuracy: 1.0000 - val_loss: 0.0049
Epoch 5/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 8s 61ms/step - accuracy: 1.0000 - loss: 0.0039 - val_accuracy: 1.0000 - val_loss: 0.0029
Epoch 6/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 8s 61ms/step - accuracy: 1.0000 - loss: 0.0024 - val_accuracy: 1.0000 - val_loss: 0.0020
Epoch 7/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 8s 61ms/step - accuracy: 1.0000 - loss: 0.0017 - val_accuracy: 1.0000 - val_loss: 0.0015
Epoch 8/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 8s 61ms/step - accuracy: 1.0000 - loss: 0.0013 - val_accuracy: 1.00

In [4]:
import numpy as np
import tensorflow as tf

# ==========================================
# 🎯 纯净版：1:1 还原训练环境
# ==========================================

# 核心机密：直接从内存中读取训练时的标准装甲长度！
# (假设您刚才训练用的 X_enc 还在内存中)
VALID_ENC_LEN = X_enc.shape[1] 

def translate_date_ultimate(date_string):
    print(f"📡 接收到未知情报: '{date_string}'")
    
    # 1. 编码器：必须且只能补齐到 VALID_ENC_LEN！
    # 多了少了这个宇宙的引力就不对！
    enc_input = [char_to_id.get(c, 0) for c in date_string] 
    enc_input_arr = tf.keras.preprocessing.sequence.pad_sequences(
        [enc_input], maxlen=VALID_ENC_LEN, padding="post"
    )
    
    # 2. 解码器：保持纯净的自回归生长
    dec_input = [char_to_id["<sos>"]]
    
    translated_date = ""
    for i in range(10): # 目标日期长度恒为 10
        dec_input_arr = np.array([dec_input]) 
        
        predictions = transformer_model.predict([enc_input_arr, dec_input_arr], verbose=0)
        
        # 永远只取最后一步的推演结果
        predicted_id = np.argmax(predictions[0, -1, :])
        
        if predicted_id == char_to_id["<pad>"]:
            break
            
        predicted_char = id_to_char[predicted_id]
        translated_date += predicted_char
        
        dec_input.append(predicted_id)

    print(f"✅ 破解翻译结果: '{translated_date}'\n")
    return translated_date

# 💥 战术重演：最高规格检阅！
translate_date_ultimate("May 15, 2049")
translate_date_ultimate("15 May 2049")
translate_date_ultimate("05/15/2049")
translate_date_ultimate("15-05-2049")

📡 接收到未知情报: 'May 15, 2049'
✅ 破解翻译结果: '2049-05-15'

📡 接收到未知情报: '15 May 2049'
✅ 破解翻译结果: '2049-05-15'

📡 接收到未知情报: '05/15/2049'
✅ 破解翻译结果: '2049-05-15'

📡 接收到未知情报: '15-05-2049'
✅ 破解翻译结果: '2049-05-15'



'2049-05-15'